# SubLens · OCR phụ đề bằng GPU Colab

Đây là luồng riêng với app Windows: chọn vùng trong app → xuất gói → quét ở đây → nhập kết quả về app.
Trong app chọn **Colab GPU → 1. Xuất gói chạy Colab** để lấy `job.zip`.

Chọn **Runtime → Change runtime type → GPU** (ví dụ T4 nếu được cấp), rồi chạy từng ô từ trên xuống.
Notebook yêu cầu CUDA thực sự hoạt động; sẽ báo lỗi nếu engine chỉ dùng CPU.
Cài đặt lần đầu có thể mất vài phút và tải các thư viện CUDA/cuDNN.
Video được xử lý trên máy chủ Google. Giữ notebook mở khi quét;
Colab có thể ngắt phiên hoặc không cấp GPU. Notebook không tự khôi phục một lượt quét bị ngắt.
[Giới hạn Colab](https://research.google.com/colaboratory/faq.html).

In [ ]:

# @title 1. Kiểm tra GPU và cài thư viện
import subprocess
import sys

gpu = subprocess.run(
    ['nvidia-smi'],
    capture_output=True,
    text=True
)

if gpu.returncode:
    raise RuntimeError(
        'Chưa có GPU. Chọn Runtime → Change runtime type → GPU.'
    )

print(gpu.stdout)
print("Python:", sys.version)

def pip(*args):
    subprocess.run(
        [sys.executable, '-m', 'pip', *args],
        check=True
    )

# Loại bỏ các phiên bản ONNX Runtime cũ
pip('uninstall', '-y',
    'rapidocr-onnxruntime',
    'onnxruntime',
    'onnxruntime-gpu')

# Cài RapidOCR mới, không ép NumPy xuống 1.x
pip('install', '-U',
    'rapidocr',
    'opencv-python-headless',
    'av',
    'Pillow')

# Cài ONNX Runtime GPU cùng CUDA/cuDNN
pip('install', '-U',
    'onnxruntime-gpu[cuda,cudnn]==1.23.2')

# Kiểm tra GPU Execution Provider
import onnxruntime as ort

print("ONNX Runtime:", ort.__version__)
print("Providers:", ort.get_available_providers())

if 'CUDAExecutionProvider' in ort.get_available_providers():
    print("CUDA Execution Provider đã có sẵn.")
else:
    print("Cảnh báo: ONNX Runtime chưa nhận CUDA Provider.")

print("Cài đặt hoàn tất.")

In [ ]:
# @title 2. Tải job.zip xuất từ app
from google.colab import files
from pathlib import Path
import importlib
import json
import tempfile
import zipfile

job = cfg = video = output = None
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Chỉ chọn một tệp job.zip xuất từ app.')
bundle = Path(next(iter(uploaded)))
del uploaded
required = {'job.json', 'subocr/__init__.py', 'subocr/core.py', 'subocr/video.py',
            'subocr/editing.py', 'subocr/colab.py', 'colab/run_job.py'}
allowed = required | {'models/en_PP-OCRv4_rec_mobile.onnx'}
workspace = Path(tempfile.mkdtemp(prefix='sublens_', dir='/content'))
with zipfile.ZipFile(bundle) as archive:
    names = archive.namelist()
    if len(set(names)) != len(names) or not required <= set(names) or not set(names) <= allowed:
        raise ValueError('Gói không đúng định dạng. Hãy xuất lại từ app SubLens.')
    if sum(info.file_size for info in archive.infolist()) > 100 * 1024 * 1024:
        raise ValueError('Gói quá lớn. Video phải tải riêng, không nằm trong job.zip.')
    for name in names:
        archive.extract(name, workspace)

# Reload only this package if a different job was uploaded in the same notebook session.
for name in list(sys.modules):
    if name == 'subocr' or name.startswith('subocr.'):
        del sys.modules[name]
sys.path.insert(0, str(workspace))
importlib.invalidate_caches()
from subocr.colab import read_json, validate_job, verify_video
job = read_json(workspace / 'job.json')
cfg = validate_job(job)
print('Video gốc:', job['video']['name'])
print('Ngôn ngữ:', cfg.language, '· vùng:', cfg.roi)
print('Gói đã sẵn sàng. Chọn nguồn video trong ô 3.')

## Chọn video
Với video lớn, nên dùng Google Drive: tải video gốc lên Drive, chọn **Google Drive** ở ô dưới
và điền đường dẫn, ví dụ `/content/drive/MyDrive/video.mp4`.
Với video nhỏ có thể chọn **Tải lên**. Dùng đúng tệp gốc, chưa cắt hoặc chuyển mã;
tên tệp có thể khác. Notebook kiểm tra dung lượng và dấu vân tay lấy mẫu đầu/giữa/cuối tệp.

In [ ]:
# @title 3. Chọn video gốc
video_source = "Google Drive" # @param ["Google Drive", "Tải lên"]
drive_video_path = "/content/drive/MyDrive/video.mp4" # @param {type:"string"}
copy_to_local = True # @param {type:"boolean"}

import shutil
video = output = None
if video_source == 'Google Drive':
    from google.colab import drive
    drive.mount('/content/drive')
    video = Path(drive_video_path.strip())
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Chỉ chọn một video gốc.')
    video = Path(next(iter(uploaded))).resolve()
    del uploaded
if not video.is_file():
    raise ValueError('Không tìm thấy video. Kiểm tra đường dẫn ở trên.')
verify_video(job, video)
if video_source == 'Google Drive' and copy_to_local:
    # Sequential decoding from the local disk avoids repeated Drive reads.
    if shutil.disk_usage(workspace).free < video.stat().st_size + 256 * 1024 * 1024:
        raise ValueError('Không đủ dung lượng tạm. Tắt copy_to_local hoặc chọn video nhỏ hơn.')
    local_video = workspace / ('input_video' + video.suffix)
    shutil.copyfile(video, local_video)
    video = local_video
print('Đã kiểm tra đúng video:', video)

In [ ]:
# @title 4. Quét OCR bằng GPU
save_results_to_drive = True # @param {type:"boolean"}

# A new output directory for every run prevents an older result being downloaded after an error.
output = None
run_output = Path(tempfile.mkdtemp(prefix='result_', dir=workspace))
command = [sys.executable, str(workspace / 'colab/run_job.py'),
           '--job', str(workspace / 'job.json'), '--video', str(video), '--output', str(run_output)]
# The child streams progress and checks CUDA on the actual OCR sessions.
subprocess.run(command, check=True)
output = run_output
if save_results_to_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/SubLens') / job['job_id'] / output.name
    destination.mkdir(parents=True, exist_ok=True)
    for source in output.iterdir():
        shutil.copyfile(source, destination / source.name)
    print('Đã lưu kết quả lên Drive:', destination)
print('Quét xong. Chạy ô 5 để tải kết quả về máy.')

In [ ]:
# @title 5. Tải kết quả về app
if globals().get('output') is None or not (output / 'result.sublens.json').is_file():
    raise RuntimeError('Chưa có kết quả. Chạy thành công ô 4 trước.')
files.download(str(output / 'result.sublens.json'))
print('Trong app: mở video gốc → Colab GPU → 2. Nhập kết quả Colab → chọn result.sublens.json.')
print('Sau đó sửa chữ, thời gian, xóa/gộp, hoàn tác và xuất SRT như bình thường.')
print('Bản SRT cũng nằm tại:', output / ('result.' + cfg.language + '.srt'))